# Tutoriel d'arbre généalogique avec UnifyWeaver

Ce carnet interactif montre comment utiliser UnifyWeaver pour compiler des prédicats Prolog en scripts Bash.

## Prérequis

- SWI-Prolog installé
- Bibliothèque UnifyWeaver disponible
- Noyau Jupyter Prolog installé (`pip install prolog-jupyter-kernel`)

## Objectifs d'apprentissage

À la fin de ce carnet, vous saurez :
1. Définir des faits et règles Prolog
2. Utiliser UnifyWeaver pour compiler des prédicats en Bash
3. Tester les scripts Bash générés
4. Comprendre la compilation de fermeture transitive

## Étape 1 : Initialiser l'environnement UnifyWeaver

Tout d'abord, nous devons charger les modules UnifyWeaver. Nous utiliserons le fichier `init.pl` du répertoire education.

In [ ]:
% Charger le fichier d'initialisation
['../init'].

## Étape 2 : Définir les relations familiales

Définissons quelques relations parent-enfant à partir de l'arbre généalogique biblique.

In [ ]:
% Définir les faits parent
:- dynamic parent/2.

parent(abraham, isaac).
parent(abraham, ishmael).
parent(isaac, jacob).
parent(isaac, esau).
parent(jacob, reuben).
parent(jacob, simeon).
parent(jacob, levi).
parent(jacob, judah).

## Étape 3 : Tester les requêtes de parenté

Avant de compiler, vérifions que nos données sont correctes à l'aide de quelques requêtes Prolog.

In [ ]:
% Requête : Qui sont les enfants d'Abraham ?
parent(abraham, Child).

In [ ]:
% Requête : Qui sont les enfants de Jacob ?
parent(jacob, Child).

## Étape 4 : Définir la relation d'ancêtre

Définissons maintenant la fermeture transitive — la relation `ancestor`.

In [ ]:
% Définir ancestor comme la fermeture transitive de parent
:- dynamic ancestor/2.

% Cas de base : un parent est un ancêtre
ancestor(X, Y) :- parent(X, Y).

% Cas récursif : si X est le parent de Y et Y est l'ancêtre de Z, alors X est l'ancêtre de Z
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

## Étape 5 : Tester les requêtes d'ancêtre

Vérifions que notre prédicat d'ancêtre fonctionne correctement.

In [ ]:
% Requête : Abraham est-il un ancêtre de Jacob ?
( ancestor(abraham, jacob) ->
    writeln('Yes: Abraham is an ancestor of Jacob')
;
    writeln('No: Abraham is not an ancestor of Jacob')
).

In [ ]:
% Requête : Quels sont tous les descendants d'Abraham ?
ancestor(abraham, Descendant).

## Étape 6 : Compiler Parent en Bash

Place à la partie pratique — compilons nos faits `parent/2` en un script Bash !

In [ ]:
% Charger le compilateur de flux
\+ \+ (
    use_module(unifyweaver(core/stream_compiler)),

    % Compiler les faits parent en bash
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    writeln('Generated Bash code for parent/2:'),
    writeln(_BashCode)
).

## Étape 7 : Enregistrer le script de Parent

Enregistrons le code Bash généré dans un fichier.

In [ ]:
% Enregistrer dans un fichier
\+ \+ (
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    setup_call_cleanup(
        open('../output/parent.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/parent.sh')
).

## Étape 8 : Compiler Ancestor en Bash

Compilons maintenant le prédicat `ancestor/2`, qui utilise la récursion.

In [ ]:
% Charger le compilateur récursif
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),

    % Compiler ancestor en bash
    compile_recursive(ancestor/2, [], _BashCode),
    writeln('Generated Bash code for ancestor/2:'),
    writeln(_BashCode)
).

## Étape 9 : Enregistrer le script d'Ancestor

Enregistrez le script d'ancêtre dans un fichier.

In [ ]:
% Enregistrer dans un fichier
\+ \+ (
    compile_recursive(ancestor/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/ancestor.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/ancestor.sh')
).

## Étape 10 : Tester les scripts générés

Testons maintenant nos scripts Bash générés ! Nous utiliserons la commande magique `%%bash` pour exécuter des commandes bash.

In [ ]:
%%bash
# Charger le script parent avec source
source ../output/parent.sh

# Test : Qui sont les enfants d'Abraham ?
echo "Enfants d'Abraham :"
parent abraham

In [ ]:
%%bash
# Charger les deux scripts avec source
source ../output/parent.sh
source ../output/ancestor.sh

# Test : Qui sont les descendants d'Abraham ?
echo "Descendants d'Abraham :"
ancestor abraham

In [ ]:
%%bash
# Charger les deux scripts avec source
source ../output/parent.sh
source ../output/ancestor.sh

# Test : Abraham est-il un ancêtre de Juda ?
if ancestor abraham judah >/dev/null 2>&1; then
    echo "✓ Oui, Abraham est un ancêtre de Juda"
else
    echo "✗ Faux"
fi

## Étape 11 : Comprendre la stratégie de compilation

Analysons ce que UnifyWeaver a accompli :

1. **Compilation de parent** : A utilisé `stream_compiler` pour créer une fonction de flux simple qui renvoie toutes les paires parent-enfant

2. **Compilation d'ancestor** : A détecté le motif de fermeture transitive et appliqué l'optimisation BFS (recherche en largeur) pour calculer efficacement tous les ancêtres accessibles

Vérifions la stratégie de compilation :

In [ ]:
% Vérifier si ancestor est classé comme récursif
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),
    recursive_compiler:classify_predicate(ancestor/2, _Classification),
    format('Ancestor classification: ~w~n', [_Classification])
).

## Résumé

Dans ce carnet, vous avez appris :

✅ Comment définir des faits et règles Prolog

✅ Comment utiliser le `stream_compiler` d'UnifyWeaver pour les faits

✅ Comment utiliser le `recursive_compiler` d'UnifyWeaver pour les prédicats récursifs

✅ Comment tester les scripts Bash générés

✅ Qu'UnifyWeaver détecte automatiquement la fermeture transitive et applique l'optimisation BFS

## Prochaines étapes

Essayez ces exercices :

1. Ajouter d'autres membres de la famille à l'arbre
2. Définir un prédicat `grandparent/2` et le compiler
3. Créer un prédicat `sibling/2` (deux personnes ayant le même parent)
4. Explorer le code Bash généré pour comprendre le fonctionnement de l'algorithme BFS

Passez au **Carnet 2 : Comparaison des motifs de récursion** pour en savoir plus sur les motifs de récursion avancés !